# SolarCast - NOAA HRRR Forecast Weather Extraction (Google Colab)

This notebook executes the Mode F1 HRRR forecast feature extraction pipeline on Google Colab.
Cloud infrastructure provides ~50-100x higher throughput and lower latency to AWS S3 us-west-1
compared to local residential connections.

### Setup Instructions:
1. Upload the folder `SolarCast_inputs` (from `D:\SolarCast_inputs` on your laptop) to your Google Drive root folder:
   `My Drive / SolarCast_inputs /` containing:
   - `hourly.parquet`
   - `hrrr_coverage_scan.parquet`
2. Run the cells in order.
3. Extracted monthly parts are saved directly to Google Drive at `My Drive / SolarCast_parts/` so progress is persisted across disconnections.


In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Cell 2: Clone repository and checkout branch
%cd /content
!rm -rf SolarCast
!git clone https://github.com/AwanTech095/SolarCast.git
%cd /content/SolarCast
!git checkout m1/hrrr-forecast-features


In [ ]:
# Cell 3: Install extraction dependencies
!pip install -q -r requirements-extract.txt


In [ ]:
# Cell 4: Copy input files from Google Drive to local repository
!mkdir -p data/processed
!cp /content/drive/MyDrive/SolarCast_inputs/hourly.parquet data/processed/
!cp /content/drive/MyDrive/SolarCast_inputs/hrrr_coverage_scan.parquet data/processed/
!ls -lh data/processed/


In [ ]:
# Cell 5: Self-test - extract 2022-06-13 00:00 UTC and verify against reference
import numpy as np
import pandas as pd
from src.data.hrrr import (
    HRRRExtractorConfig,
    extract_hour_weather,
    get_thread_fs,
    load_coverage_scan_set,
)

cfg = HRRRExtractorConfig(variables=["tmp", "tcdc"])
fs = get_thread_fs()
scan_set = load_coverage_scan_set("data/processed/hrrr_coverage_scan.parquet")

t = pd.Timestamp("2022-06-13 00:00:00", tz="UTC")
res = extract_hour_weather(t, cfg, fs, scan_set)

print("Self-test extraction result:")
for k, v in res.items():
    print(f"  {k}: {v}")

REF_TMP_C = 25.6
assert np.isclose(res["tmp_c"], REF_TMP_C, atol=0.1), f"Expected {REF_TMP_C} C, got {res['tmp_c']}"
assert res["extraction_status"].startswith(("SUCCESS", "FALLBACK_SUCCESS"))
print("
[PASSED] Self-test successfully verified against reference value!")


In [ ]:
# Cell 6: Run extraction with configurable start/end dates
# Priority 1: 2022-10-17 to 2023-11-09 (Test & latest Dev fold)
# Priority 2: 2021-10-17 to 2022-10-16 (Prior training fold)
# Priority 3: Older months as time permits
START_DATE = "2022-10-17"
END_DATE = "2023-11-09"
VARIABLES = "tmp,tcdc"
THREADS = 16
PARTS_DIR = "/content/drive/MyDrive/SolarCast_parts"

!mkdir -p {PARTS_DIR}
!python scripts/hrrr_extract.py \
    --start {START_DATE} \
    --end {END_DATE} \
    --variables {VARIABLES} \
    --threads {THREADS} \
    --parts-dir {PARTS_DIR}


In [ ]:
# Cell 7: Check live progress and inspect completed parts on Google Drive
!cat logs/hrrr_extract.log | tail -n 15
!ls -lh /content/drive/MyDrive/SolarCast_parts/


In [ ]:
# Cell 8: Merge completed parts into final consolidated dataset on Drive
PARTS_DIR = "/content/drive/MyDrive/SolarCast_parts"
MERGED_FILE = "/content/drive/MyDrive/SolarCast_parts/hrrr_forecast_features.parquet"

!python scripts/hrrr_extract.py \
    --parts-dir {PARTS_DIR} \
    --merged-file {MERGED_FILE} \
    --merge

import pandas as pd
df = pd.read_parquet(MERGED_FILE)
print(f"Merged records count: {len(df):,}")
print(df.info())
print(df.head())
